In [9]:
import os
import gc

import numpy as np
import pandas as pd

from transformers import pipeline
from transformers import AutoTokenizer
from transformers import DataCollatorForLanguageModeling
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

import datasets
from datasets import ClassLabel, load_dataset, Dataset, DatasetDict, load_metric

import warnings
warnings.filterwarnings('ignore')

In [23]:
class CFG:
    wandb = False
    report_to = None
    lab_assignment = 5
    _wandb_kernel = "temuujin"

    debug = False
    num_workers = 12

    tokenizer_name = 'bayartsogt/mongolian-gpt2'
    model_name = 'bayartsogt/mongolian-gpt2'

    project = 'NUM-Machine-Learning-Lab-5'
    name = "Lab 5"

    config = {
        "output_dir": "lab5_finetune_gpt2",
        "group": model_name,
        "learning_rate": 1e-5,
        "weight_decay": 1e-3,
        'num_train_epochs': 50,
        "train_batch_size": 32,
        "eval_batch_size": 32,
        "dataloader_num_workers": num_workers,
        "finetuning_task": 'ner',
        "evaluation_strategy": 'epoch',
        "logging_strategy": 'epoch',
        "overwrite_output_dir": True,
        "push_to_hub": False,
    }

    model_save_dir = "lab5_gpt2"

    test_size = 0.2

    train = True
    eval = True

    eval_metric = "seqeval"

    early_stopping_patience = 15

if CFG.debug:
    CFG.config['num_train_epochs'] = 2

if CFG.wandb:
    os.environ["WANDB_SILENT"] = "True"
    CFG.report_to = "wandb"

    import wandb
    wandb.login()

    run = wandb.init(
        project = CFG.project,
        name = CFG.name,
        config = CFG.config
    )

config = CFG.config

In [11]:
def concatenate_columns(example):
    example["prompt"] = example["prompt"] + " " + example["answer"]
    return example

In [12]:
df = pd.read_csv('khk.noun.tsv', sep = '\t')

df['prompt'] = '<s> bb: ' + df['prompt']
df['answer'] = df['answer'] + '</s>'

infl_dataset = Dataset.from_pandas(df)
ds_train_devtest = infl_dataset.train_test_split(test_size = 0.025, seed = 42)
ds_devtest = ds_train_devtest['test'].train_test_split(test_size = 0.5, seed = 42)

ds_splits = DatasetDict({
    'train': ds_train_devtest['train'],
    'valid': ds_devtest['train'],
    'test': ds_devtest['test']
})

ds_splits["train"] = ds_splits["train"].map(concatenate_columns)
ds_splits["valid"] = ds_splits["valid"].map(concatenate_columns)
ds_splits["test"] = ds_splits["test"].map(concatenate_columns)

ds_splits = ds_splits.flatten()

Map:   0%|          | 0/14036 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

In [13]:
block_size = 64
tokenizer = AutoTokenizer.from_pretrained(CFG.tokenizer_name)

def preprocess_function(examples):
    return tokenizer(examples["prompt"])

tokenized_ds = ds_splits.map(
    preprocess_function,
    batched = True,
    num_proc = 4,
    remove_columns = ds_splits["train"].column_names,
)

Map (num_proc=4):   0%|          | 0/14036 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

In [14]:
def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_dataset = tokenized_ds.map(group_texts, batched = True, num_proc = 4)

Map (num_proc=4):   0%|          | 0/14036 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

In [24]:
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer = tokenizer, mlm = False)
model = AutoModelForCausalLM.from_pretrained(CFG.model_name)

config = CFG.config

test_ver = 1
OUTPUT_MODEL = os.path.join(CFG.model_save_dir, f"test_v{test_ver}")

training_args = TrainingArguments(
    report_to = CFG.report_to,
    output_dir = OUTPUT_MODEL,
    num_train_epochs = config["num_train_epochs"],
    per_device_train_batch_size = config["train_batch_size"],
    per_device_eval_batch_size = config["eval_batch_size"],
    overwrite_output_dir = config["overwrite_output_dir"],
    learning_rate = config["learning_rate"],
    weight_decay = config["weight_decay"],
    evaluation_strategy = config["evaluation_strategy"],
    push_to_hub = config["push_to_hub"],
    do_eval = True,
    disable_tqdm = True
)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = lm_dataset["train"],
    eval_dataset = lm_dataset["test"],
    tokenizer = tokenizer,
    data_collator = data_collator,
)

trainer.train()

{'eval_loss': 4.76384973526001, 'eval_runtime': 0.0545, 'eval_samples_per_second': 587.371, 'eval_steps_per_second': 18.355, 'epoch': 1.0}
{'eval_loss': 3.034586191177368, 'eval_runtime': 0.0549, 'eval_samples_per_second': 582.461, 'eval_steps_per_second': 18.202, 'epoch': 2.0}
{'eval_loss': 2.051705837249756, 'eval_runtime': 0.0548, 'eval_samples_per_second': 583.548, 'eval_steps_per_second': 18.236, 'epoch': 3.0}
{'eval_loss': 1.7107183933258057, 'eval_runtime': 0.0549, 'eval_samples_per_second': 583.276, 'eval_steps_per_second': 18.227, 'epoch': 4.0}
{'eval_loss': 1.575294852256775, 'eval_runtime': 0.0547, 'eval_samples_per_second': 584.516, 'eval_steps_per_second': 18.266, 'epoch': 5.0}
{'eval_loss': 1.506796956062317, 'eval_runtime': 0.0549, 'eval_samples_per_second': 582.542, 'eval_steps_per_second': 18.204, 'epoch': 6.0}
{'loss': 2.9889, 'grad_norm': 0.6968111991882324, 'learning_rate': 8.765432098765432e-06, 'epoch': 6.17}
{'eval_loss': 1.46463942527771, 'eval_runtime': 0.0547,

TrainOutput(global_step=4050, training_loss=1.2207829896903333, metrics={'train_runtime': 758.7181, 'train_samples_per_second': 170.155, 'train_steps_per_second': 5.338, 'train_loss': 1.2207829896903333, 'epoch': 50.0})

In [25]:
eval_results = trainer.evaluate()

{'eval_loss': 1.5303469896316528, 'eval_runtime': 0.0687, 'eval_samples_per_second': 465.773, 'eval_steps_per_second': 14.555, 'epoch': 50.0}


In [28]:
prompt = '<s> bb: мангарын'
generator = pipeline("text-generation", model = model, tokenizer = tokenizer, num_beams = 5)

ans = generator(prompt, max_length = 20)

print(str(ans[0]['generated_text']))
print(str(ans))

<s> bb: мангарын мануулагчтай мангаар ман <ins> bb: банз
[{'generated_text': '<s> bb: мангарын мануулагчтай мангаар ман <ins> bb: банз'}]
